# 🚀 一键训练流程 - RAG Email System

## 功能

这个 notebook 完成完整的训练和部署流程：

1. ✅ 挂载 Google Drive + Git Pull 最新代码
2. ✅ 加载数据（hospital & corruption）
3. ✅ 构建**基础模型**索引 → 保存到 Drive
4. ✅ 训练**微调模型**（LoRA）
5. ✅ 构建**微调模型**索引 → 保存到 Drive
6. ✅ 推送所有内容到 HuggingFace Hub

**运行环境**: Google Colab with GPU (T4/V100)

**一键运行**: 点击 Runtime → Run all

---

## Step 0: Environment Setup

In [ ]:
# 检查 GPU
import torch
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU detected - training will be slow!")

In [ ]:
# 安装依赖
print("📦 Installing dependencies...\n")

!pip install -q sentence-transformers faiss-cpu pyyaml tqdm
!pip install -q peft datasets accelerate
!pip install -q huggingface_hub

print("✅ Dependencies installed")

## Step 1: Mount Drive & Pull Code

In [ ]:
# 挂载 Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
print("✅ Google Drive mounted")

# 切换到工作目录
os.chdir('/content')

In [ ]:
# Clone/Pull 最新代码
REPO_URL = "https://github.com/Echo-Lee/RAG-embedding.git"
REPO_NAME = "RAG-embedding"

if os.path.exists(f'/content/{REPO_NAME}'):
    print(f"📥 Pulling latest changes...")
    !cd /content/{REPO_NAME} && git pull
else:
    print(f"📦 Cloning repository...")
    !git clone {REPO_URL} /content/{REPO_NAME}

os.chdir(f'/content/{REPO_NAME}')
print(f"\n✅ Code ready at: {os.getcwd()}")

# Add to Python path
import sys
sys.path.insert(0, 'src')
print("✅ Python path configured")

## Step 2: Configuration

In [ ]:
from pathlib import Path
import json

# ========== 配置 ==========

# Google Drive 路径
DRIVE_BASE = Path("/content/drive/MyDrive/Epiq Project/pipeline")

# 数据路径
DATA_BASE = DRIVE_BASE / "data/processed"
DATA_PATHS = {
    'hospital': DATA_BASE / "hospital/threads_with_summary.json",
    'corruption': DATA_BASE / "corruption/emails_group_by_thread.json"
}

# 输出路径
INDEX_BASE = DRIVE_BASE / "faiss_index"
MODEL_BASE = DRIVE_BASE / "models"

# 确保目录存在
INDEX_BASE.mkdir(parents=True, exist_ok=True)
MODEL_BASE.mkdir(parents=True, exist_ok=True)

# 模型配置
BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"
BATCH_SIZE = 32

# HuggingFace 配置
HF_USERNAME = "ChenyuEcho"
HF_INDEX_REPO = f"{HF_USERNAME}/rag-indexes"
HF_MODEL_REPO_BASE = f"{HF_USERNAME}/rag-finetuned"  # Will append -hospital or -corruption

# 训练配置
TRAIN_DATASETS = ['hospital', 'corruption']  # 选择要训练的数据集
TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
LORA_R = 16
LORA_ALPHA = 32

# ========== 配置完成 ==========

print("📁 Configuration:")
print(f"  Data: {DATA_BASE}")
print(f"  Indexes: {INDEX_BASE}")
print(f"  Models: {MODEL_BASE}")
print(f"  Base Model: {BASE_MODEL}")
print(f"  HF Username: {HF_USERNAME}")
print(f"  Train Datasets: {TRAIN_DATASETS}")

## Step 3: Verify Data

In [ ]:
# 检查数据文件
print("🔍 Checking data files...\n")

data_ok = True
for dataset, path in DATA_PATHS.items():
    if path.exists():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"  ✅ {dataset}: {path.name}")
        print(f"     Size: {size_mb:.1f} MB")
        print(f"     Path: {path}\n")
    else:
        print(f"  ❌ {dataset}: NOT FOUND")
        print(f"     Expected: {path}\n")
        data_ok = False

if not data_ok:
    print("⚠️  Please check your data paths!")
else:
    print("✅ All data files found!")

## Step 4: Load Data

In [ ]:
from data.loader import EmailDataLoader
from config.config import RAGConfig, DatasetConfig

def load_dataset(dataset_name):
    """加载指定数据集"""
    print(f"\n📥 Loading {dataset_name} dataset...")
    
    # 创建配置
    dataset_config = DatasetConfig(
        name=dataset_name,
        data_path=DATA_PATHS[dataset_name]
    )
    
    config = RAGConfig(
        dataset=dataset_config,
        embedding_model=BASE_MODEL,
        batch_size=BATCH_SIZE,
        project_root=Path.cwd()
    )
    
    # 加载数据
    loader = EmailDataLoader(config)
    documents = loader.load_documents()
    
    print(f"  ✅ Loaded {len(documents)} documents")
    if documents:
        print(f"  Sample: {documents[0].content[:100]}...")
    
    return documents, config

# 加载所有数据集
all_data = {}
for dataset_name in TRAIN_DATASETS:
    docs, cfg = load_dataset(dataset_name)
    all_data[dataset_name] = {'documents': docs, 'config': cfg}

print(f"\n✅ All datasets loaded!")

## Step 5: Build Base Model Indexes

In [ ]:
from retrieval.indexer import FAISSIndexBuilder
from sentence_transformers import SentenceTransformer
import faiss
from datetime import datetime

def build_and_save_index(dataset_name, documents, model_name_or_path, output_subdir):
    """
    构建并保存 FAISS 索引
    
    Args:
        dataset_name: 数据集名称
        documents: 文档列表
        model_name_or_path: 模型名称或路径
        output_subdir: 输出子目录名（如 'base-hospital', 'finetuned-hospital'）
    """
    print(f"\n{'='*60}")
    print(f"🔨 Building index: {output_subdir}")
    print(f"{'='*60}")
    
    # 加载模型
    print(f"📥 Loading model: {model_name_or_path}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name_or_path, device=device)
    print(f"  ✅ Model loaded on {device}")
    
    # 提取文本
    texts = [doc.content for doc in documents]
    print(f"\n🔄 Encoding {len(texts)} documents...")
    
    # 编码
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True
    )
    
    print(f"  ✅ Embeddings shape: {embeddings.shape}")
    
    # 构建 FAISS 索引
    print(f"\n🔨 Building FAISS index...")
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)  # Inner Product = cosine similarity
    index.add(embeddings)
    
    print(f"  ✅ Index built: {index.ntotal} vectors")
    
    # 保存
    output_dir = INDEX_BASE / output_subdir
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n💾 Saving to: {output_dir}")
    
    # 保存 FAISS index
    index_path = output_dir / "faiss_index.bin"
    faiss.write_index(index, str(index_path))
    print(f"  ✅ {index_path.name}")
    
    # 保存元数据
    metadata = {
        'dataset_name': dataset_name,
        'model': model_name_or_path,
        'num_docs': len(documents),
        'dimension': dimension,
        'index_type': 'IndexFlatIP',
        'created_at': datetime.now().isoformat(),
        'description': f'FAISS index for {dataset_name} using {output_subdir.split("-")[0]} model'
    }
    
    metadata_path = output_dir / "metadata.json"
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"  ✅ {metadata_path.name}")
    
    # 保存文档元数据（用于检索显示）
    doc_metadata = [
        {
            'content': doc.content,
            'metadata': doc.metadata,
            'doc_id': doc.doc_id
        }
        for doc in documents
    ]
    
    doc_meta_path = output_dir / "doc_metadata.json"
    with open(doc_meta_path, 'w', encoding='utf-8') as f:
        json.dump(doc_metadata, f, ensure_ascii=False, indent=2)
    print(f"  ✅ {doc_meta_path.name}")
    
    print(f"\n🎉 {output_subdir} complete!")
    print(f"{'='*60}\n")
    
    return output_dir

# 构建所有基础模型索引
print("\n🚀 Building BASE MODEL indexes...\n")

base_indexes = {}
for dataset_name in TRAIN_DATASETS:
    docs = all_data[dataset_name]['documents']
    output_dir = build_and_save_index(
        dataset_name=dataset_name,
        documents=docs,
        model_name_or_path=BASE_MODEL,
        output_subdir=f"base-{dataset_name}"
    )
    base_indexes[dataset_name] = output_dir

print("\n✅ All base indexes built!")

## Step 6: Prepare Training Data

In [ ]:
from sentence_transformers import InputExample
from torch.utils.data import DataLoader
import random

def prepare_training_data(documents, dataset_name):
    """
    准备训练数据：从同一个 thread 的邮件中创建 anchor-positive pairs
    """
    print(f"\n📝 Preparing training data for {dataset_name}...")
    
    # Group documents by thread_id
    thread_docs = {}
    for doc in documents:
        thread_id = doc.metadata.get('thread_id', 'unknown')
        if thread_id not in thread_docs:
            thread_docs[thread_id] = []
        thread_docs[thread_id].append(doc)
    
    print(f"  Found {len(thread_docs)} threads")
    
    # Create anchor-positive pairs
    train_examples = []
    
    for thread_id, thread_doc_list in thread_docs.items():
        if len(thread_doc_list) < 2:
            continue  # Skip threads with only one email
        
        # Create pairs from same thread (semantic similarity)
        for i in range(len(thread_doc_list)):
            for j in range(i + 1, len(thread_doc_list)):
                # Each pair has similarity score 1.0 (same thread)
                train_examples.append(
                    InputExample(
                        texts=[thread_doc_list[i].content, thread_doc_list[j].content],
                        label=1.0
                    )
                )
    
    print(f"  ✅ Created {len(train_examples)} training pairs")
    
    # Shuffle
    random.shuffle(train_examples)
    
    return train_examples

# 准备所有数据集的训练数据
training_data = {}
for dataset_name in TRAIN_DATASETS:
    docs = all_data[dataset_name]['documents']
    train_examples = prepare_training_data(docs, dataset_name)
    training_data[dataset_name] = train_examples

print("\n✅ Training data prepared!")

## Step 7: Fine-tune Models

In [ ]:
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model, TaskType

def finetune_model(base_model_name, train_examples, dataset_name):
    """
    使用 LoRA 微调模型
    """
    print(f"\n{'='*60}")
    print(f"🎯 Fine-tuning model for {dataset_name}")
    print(f"{'='*60}")
    
    # 加载基础模型
    print(f"\n📥 Loading base model: {base_model_name}")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(base_model_name, device=device)
    print(f"  ✅ Base model loaded")
    
    # 配置 LoRA
    print(f"\n🔧 Configuring LoRA...")
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "v_proj"],  # Transformer attention layers
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.FEATURE_EXTRACTION
    )
    
    # 应用 LoRA 到模型
    # Note: SentenceTransformer 的底层模型在 model[0].auto_model
    base_transformer = model[0].auto_model
    model[0].auto_model = get_peft_model(base_transformer, lora_config)
    
    print(f"  ✅ LoRA applied (r={LORA_R}, alpha={LORA_ALPHA})")
    print(f"  Trainable params: {model[0].auto_model.print_trainable_parameters()}")
    
    # 准备训练
    print(f"\n🔄 Preparing training...")
    train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=TRAIN_BATCH_SIZE)
    train_loss = losses.CosineSimilarityLoss(model)
    
    print(f"  Training examples: {len(train_examples)}")
    print(f"  Batch size: {TRAIN_BATCH_SIZE}")
    print(f"  Epochs: {TRAIN_EPOCHS}")
    print(f"  Learning rate: {LEARNING_RATE}")
    
    # 训练
    print(f"\n🚀 Starting training...\n")
    model.fit(
        train_objectives=[(train_dataloader, train_loss)],
        epochs=TRAIN_EPOCHS,
        warmup_steps=100,
        optimizer_params={'lr': LEARNING_RATE},
        show_progress_bar=True
    )
    
    print(f"\n✅ Training complete!")
    
    # 保存模型
    output_dir = MODEL_BASE / f"finetuned-{dataset_name}"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\n💾 Saving fine-tuned model to: {output_dir}")
    model.save(str(output_dir))
    
    print(f"  ✅ Model saved!")
    print(f"\n🎉 Fine-tuning complete for {dataset_name}!")
    print(f"{'='*60}\n")
    
    return output_dir

# 微调所有模型
print("\n🚀 Starting fine-tuning for all datasets...\n")

finetuned_models = {}
for dataset_name in TRAIN_DATASETS:
    train_examples = training_data[dataset_name]
    model_dir = finetune_model(BASE_MODEL, train_examples, dataset_name)
    finetuned_models[dataset_name] = model_dir

print("\n✅ All models fine-tuned!")

## Step 8: Build Fine-tuned Model Indexes

In [ ]:
# 构建所有微调模型的索引
print("\n🚀 Building FINE-TUNED MODEL indexes...\n")

finetuned_indexes = {}
for dataset_name in TRAIN_DATASETS:
    docs = all_data[dataset_name]['documents']
    model_path = str(finetuned_models[dataset_name])
    
    output_dir = build_and_save_index(
        dataset_name=dataset_name,
        documents=docs,
        model_name_or_path=model_path,
        output_subdir=f"finetuned-{dataset_name}"
    )
    finetuned_indexes[dataset_name] = output_dir

print("\n✅ All fine-tuned indexes built!")

## Step 9: Verify All Outputs

In [ ]:
# 验证所有输出
print("\n" + "="*60)
print("🔍 Verification Summary")
print("="*60 + "\n")

def verify_directory(path, name):
    """验证目录及其文件"""
    print(f"\n📁 {name}: {path}")
    if path.exists():
        files = list(path.glob('*'))
        for f in files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  ✅ {f.name} ({size_mb:.1f} MB)")
        return True
    else:
        print(f"  ❌ Directory not found!")
        return False

all_verified = True

# 验证索引
print("\n🔎 Indexes:")
for dataset_name in TRAIN_DATASETS:
    # Base index
    if not verify_directory(base_indexes[dataset_name], f"Base {dataset_name}"):
        all_verified = False
    
    # Fine-tuned index
    if not verify_directory(finetuned_indexes[dataset_name], f"Fine-tuned {dataset_name}"):
        all_verified = False

# 验证模型
print("\n🔎 Models:")
for dataset_name in TRAIN_DATASETS:
    if not verify_directory(finetuned_models[dataset_name], f"Model {dataset_name}"):
        all_verified = False

print("\n" + "="*60)
if all_verified:
    print("✅ All outputs verified!")
else:
    print("⚠️  Some outputs missing - check above")
print("="*60)

## Step 10: Upload to HuggingFace Hub

In [ ]:
from huggingface_hub import login, HfApi, create_repo

# 登录 HuggingFace
print("🔐 Login to HuggingFace Hub\n")
print("Get your token from: https://huggingface.co/settings/tokens")
print("Token should have WRITE permission\n")

hf_token = input("Enter your HF token: ")
login(token=hf_token)
print("\n✅ Logged in!\n")

api = HfApi()

In [ ]:
# 上传索引到 HF Hub
print(f"\n{'='*60}")
print(f"📤 Uploading indexes to {HF_INDEX_REPO}")
print(f"{'='*60}\n")

# 创建索引仓库
try:
    create_repo(
        repo_id=HF_INDEX_REPO,
        repo_type="dataset",
        exist_ok=True
    )
    print(f"✅ Repository created/verified: {HF_INDEX_REPO}\n")
except Exception as e:
    print(f"⚠️  Repository creation: {e}\n")

# 上传所有索引
for dataset_name in TRAIN_DATASETS:
    # Base index
    print(f"📤 Uploading base-{dataset_name}...")
    api.upload_folder(
        folder_path=str(base_indexes[dataset_name]),
        repo_id=HF_INDEX_REPO,
        repo_type="dataset",
        path_in_repo=f"base-{dataset_name}"
    )
    print(f"  ✅ Uploaded\n")
    
    # Fine-tuned index
    print(f"📤 Uploading finetuned-{dataset_name}...")
    api.upload_folder(
        folder_path=str(finetuned_indexes[dataset_name]),
        repo_id=HF_INDEX_REPO,
        repo_type="dataset",
        path_in_repo=f"finetuned-{dataset_name}"
    )
    print(f"  ✅ Uploaded\n")

print(f"🎉 All indexes uploaded!")
print(f"View at: https://huggingface.co/datasets/{HF_INDEX_REPO}\n")

In [ ]:
# 上传模型到 HF Hub
print(f"\n{'='*60}")
print(f"📤 Uploading fine-tuned models")
print(f"{'='*60}\n")

for dataset_name in TRAIN_DATASETS:
    model_repo = f"{HF_MODEL_REPO_BASE}-{dataset_name}"
    
    print(f"📤 Uploading to {model_repo}...")
    
    # 创建模型仓库
    try:
        create_repo(
            repo_id=model_repo,
            repo_type="model",
            exist_ok=True
        )
        print(f"  ✅ Repository created/verified\n")
    except Exception as e:
        print(f"  ⚠️  Repository: {e}\n")
    
    # 上传模型文件
    api.upload_folder(
        folder_path=str(finetuned_models[dataset_name]),
        repo_id=model_repo,
        repo_type="model"
    )
    print(f"  ✅ Model uploaded!")
    print(f"  View at: https://huggingface.co/{model_repo}\n")

print(f"🎉 All models uploaded!")

## 🎉 完成！

---

### ✅ 你现在有：

**在 Google Drive**:
- ✅ Base model indexes: `/Epiq Project/pipeline/faiss_index/base-{dataset}/`
- ✅ Fine-tuned indexes: `/Epiq Project/pipeline/faiss_index/finetuned-{dataset}/`
- ✅ Fine-tuned models: `/Epiq Project/pipeline/models/finetuned-{dataset}/`

**在 HuggingFace Hub**:
- ✅ Indexes: `https://huggingface.co/datasets/ChenyuEcho/rag-indexes`
- ✅ Models: `https://huggingface.co/ChenyuEcho/rag-finetuned-{dataset}`

---

### 📊 下一步：

1. **运行评估**: 打开 `colab_evaluation.ipynb` 分析模型性能

2. **部署前端**: 在 HuggingFace Space 创建 Gradio 应用
   - 使用 `gradio_app_compare.py`
   - 修改 `YOUR_HF_USERNAME = "ChenyuEcho"`
   - 添加你的模型到 `MODEL_CONFIGS`

3. **测试检索**: 继续下面的可选测试

---

## 可选：Quick Test

In [ ]:
# 快速测试：对比 base vs fine-tuned
test_dataset = TRAIN_DATASETS[0]
test_query = "What are the main issues discussed in the emails?"

print(f"🧪 Quick Test on {test_dataset}")
print(f"Query: {test_query}\n")

# 加载两个索引
base_index = faiss.read_index(str(base_indexes[test_dataset] / "faiss_index.bin"))
finetuned_index = faiss.read_index(str(finetuned_indexes[test_dataset] / "faiss_index.bin"))

# 加载模型
base_model = SentenceTransformer(BASE_MODEL)
finetuned_model_path = str(finetuned_models[test_dataset])
finetuned_model = SentenceTransformer(finetuned_model_path)

# Base model 检索
print("📊 Base Model:")
base_emb = base_model.encode([test_query], normalize_embeddings=True)
base_scores, base_indices = base_index.search(base_emb, 5)
print(f"  Top score: {base_scores[0][0]:.4f}")
print(f"  Avg score: {base_scores[0].mean():.4f}")
print(f"  Top 5 indices: {base_indices[0].tolist()}\n")

# Fine-tuned model 检索
print("📊 Fine-tuned Model:")
ft_emb = finetuned_model.encode([test_query], normalize_embeddings=True)
ft_scores, ft_indices = finetuned_index.search(ft_emb, 5)
print(f"  Top score: {ft_scores[0][0]:.4f}")
print(f"  Avg score: {ft_scores[0].mean():.4f}")
print(f"  Top 5 indices: {ft_indices[0].tolist()}\n")

# 对比
print("📈 Improvement:")
top_improvement = (ft_scores[0][0] - base_scores[0][0]) / base_scores[0][0] * 100
avg_improvement = (ft_scores[0].mean() - base_scores[0].mean()) / base_scores[0].mean() * 100
print(f"  Top score: {top_improvement:+.1f}%")
print(f"  Avg score: {avg_improvement:+.1f}%")

print("\n✅ Test complete!")